In [33]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [34]:
import importlib
from model import train as t_ 
importlib.reload(t_)

<module 'model.train' from '/home/s.senichev/anirec/model/train.py'>

In [35]:
import os, sys
import pathlib
import json

import torch
import pandas as pd
from transformers import AutoTokenizer
import torch.nn.functional as F

from data.dataset import build_id_to_text
from data.preprocessing import run_preprocessing
from model.train import (
    train_stage1, train_stage2,
    evaluate_epoch, DEFAULT_CFG, format_metrics,
    run_hpo, run_hpo_reranker, build_item_embedding_cache, tokenise_batch,
    build_cf_mappings,
)

from model.architecture import ContrastiveEncoder, TwoTowerModel
from model.reranker import CrossEncoderReranker, TwoTowerWithReranker, train_stage3
from model.metrics import evaluate_reranker, format_metrics

In [36]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = pathlib.Path('checkpoints/')
output_dir.mkdir(exist_ok=True)

ENCODER = 'intfloat/multilingual-e5-base'
use_tower_hpo = False

# Data

In [38]:
os.makedirs('raw_data', exist_ok=True)

BASE_URL = 'https://huggingface.co/datasets/kdduha/shikimori-recsys/resolve/main'

for fname in ['anime.csv', 'genres.csv', 'users_rates.csv']:
    dest = f'raw_data/{fname}'
    if not os.path.exists(dest):
        !wget -q "{BASE_URL}/{fname}" -O "{dest}"
        print(f'Downloaded {fname}')
    else:
        print(f'Already have {fname}')

!ls -lh raw_data/

Already have anime.csv
Already have genres.csv
Already have users_rates.csv


total 15M
-rw-rw-r-- 1 s.senichev s.senichev 8.7M Mar  2 23:30 anime.csv
-rw-rw-r-- 1 s.senichev s.senichev 2.9K Mar  2 23:30 genres.csv
-rw-rw-r-- 1 s.senichev s.senichev 6.0M Mar  2 23:30 users_rates.csv


In [39]:
stats = run_preprocessing(
    data_dir='raw_data',
    output_dir='processed_data',
)
print(json.dumps(stats, indent=2))

02:09:16  INFO      Loading raw CSVs from raw_data
02:09:16  INFO      Raw sizes — anime: 9950  genres: 80  interactions: 67071
02:09:18  INFO      Anime processing done. Non-null text_input: 9950 / 9950
02:09:18  INFO      Saved anime_processed.parquet
02:09:18  INFO      Saved genre_vocab.json  (80 genres)
02:09:19  INFO      Dropped 0 rows with unparseable anime_id
02:09:19  INFO      Interactions — explicit (scored): 28129  implicit (watched, unscored): 38677  dropped (score=0, episodes=0): 265
02:09:20  INFO      Promoted 38666 implicit interactions (completion >= 80%) to explicit score=7
02:09:20  INFO      Dropped 1515 interactions for unknown anime_id
02:09:20  INFO      Removed users with < 2 total interactions (explicit+implicit). Kept: 993 users  65143 rows
02:09:20  INFO      Final — 65143 rows (65132 explicit, 11 implicit)  993 users  4182 items
02:09:20  INFO      Temporal split — train: 63353  val: 895  test: 895  (eligible users: 895 / 993)
02:09:20  WARNING   Timestamp

{
  "n_anime_raw": 9950,
  "n_anime_processed": 9950,
  "n_genres": 80,
  "n_interactions_raw": 67071,
  "n_interactions_clean": 65143,
  "n_users": 993,
  "n_items": 4182,
  "train_size": 63353,
  "val_size": 895,
  "test_size": 895,
  "score_mean": 7.401,
  "score_std": 1.215,
  "density_pct": 1.5687
}


In [40]:
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
val_df = pd.read_parquet('processed_data/val_interactions.parquet')
test_df = pd.read_parquet('processed_data/test_interactions.parquet')

print('Anime sample')
display(anime_df[['id','name','genre_names','rating_ordinal','score_global','text_input']].head(3))

print('\nInteraction sample')
display(train_df.head(3))

print(f'\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
# print(f'Score distribution:\n{train_df.score_raw.value_counts().sort_index()}')

Anime sample


,id,name,genre_names,rating_ordinal,score_global,text_input
0,52991,Sousou no Frieren,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.921111,Sousou no Frieren (Провожающая в последний пут...
1,59978,Sousou no Frieren 2nd Season,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.910000,Sousou no Frieren 2nd Season (Провожающая в по...
2,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy, Shounen, M...",3.0,0.900000,Fullmetal Alchemist: Brotherhood (Стальной алх...



Interaction sample


,user_id,anime_id,score_raw,score_norm,rewatches,episodes,completion_rate,confidence,is_explicit,created_at
0,1721273,5114,9,0.888889,0,64,1.0,0.888889,True,2025-11-07 19:14:34+00:00
1,1721273,9253,10,1.000000,0,24,1.0,1.000000,True,2025-11-07 19:15:22+00:00
2,1721273,2001,10,1.000000,0,27,1.0,1.000000,True,2025-11-07 19:18:22+00:00



Train: 63,353  Val: 895  Test: 895


# Model

### Load checkpoint if it exists

In [41]:
# output_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')

# with open(output_dir / 'config.json') as f:
#     cfg = json.load(f)

# tokenizer = AutoTokenizer.from_pretrained(str(output_dir))

# # Rebuild CF mappings
# item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
# cf_dim = cfg.get('cf_dim', 0)
# if cf_dim == 0:
#     n_items_cf = 0
#     n_users_cf = 0

# model = TwoTowerModel(
#     encoder_name=ENCODER,
#     proj_dim=cfg['proj_dim'],
#     nhead=cfg['nhead'],
#     temperature=cfg['temperature'],
#     dropout=cfg['dropout'],
#     freeze_mode=cfg['freeze_mode'],
#     lora_rank=cfg['lora_rank'],
#     lora_alpha=cfg.get('lora_alpha', 16.0),
#     n_items=n_items_cf,
#     n_users=n_users_cf,
#     cf_dim=cf_dim,
#     user_tower_layers=cfg.get('user_tower_layers', 1),
# )

# state = torch.load(output_dir / 'model.pt', map_location=device)
# model.load_state_dict(state, strict=False)
# model.to(device)
# model.eval()
# print('Final model loaded')

## Train main model

### HPO

In [42]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
hpo_path = output_dir / 'hpo_best_params.json'

if use_tower_hpo:
    print('Starting HPO')
    best_params = run_hpo(
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=DEFAULT_CFG,
        output_dir=pathlib.Path('checkpoints'),
        device=device,
        n_trials=20,
        encoder_name=ENCODER,
    )
    
    with open(hpo_path) as f:
        hpo_data = json.load(f)
    cfg = {**DEFAULT_CFG, **hpo_data['best_params']}
    print(f'Using HPO params (NDCG@10={hpo_data["best_value"]:.4f})')

else:
    cfg = {**DEFAULT_CFG}
    print('Using default hyperparameters')

print(json.dumps(cfg, indent=2))

02:09:21  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02:09:21  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/835193815a3936a24a0ee7dc9e3d48c1fbb19c55/config.json "HTTP/1.1 200 OK"
02:09:21  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
02:09:21  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/835193815a3936a24a0ee7dc9e3d48c1fbb19c55/tokenizer_config.json "HTTP/1.1 200 OK"
02:09:21  INFO      HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
02:09:21  INFO      HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-base/tree/mai

Using default hyperparameters
{
  "s1_epochs": 7,
  "s1_batch_size": 32,
  "s1_grad_accum": 4,
  "s1_lr": 2e-05,
  "s1_warmup_steps": 100,
  "s1_max_length": 512,
  "s1_neg_per_user": 5,
  "s1_grad_clip": 1.0,
  "s2_epochs": 10,
  "s2_batch_size": 64,
  "s2_grad_accum": 2,
  "s2_lr": 3e-05,
  "s2_warmup_steps": 200,
  "s2_max_length": 512,
  "s2_max_history": 50,
  "s2_grad_clip": 1.0,
  "s2_encode_batch": 128,
  "s2_cache_refresh_steps": 50,
  "proj_dim": 256,
  "nhead": 4,
  "temperature": 0.07,
  "dropout": 0.1,
  "weight_decay": 0.01,
  "lora_rank": 8,
  "lora_alpha": 16.0,
  "lora_dropout": 0.05,
  "freeze_mode": "lora",
  "encoder": "intfloat/multilingual-e5-base",
  "pooling": "mean",
  "cf_dim": 128,
  "user_tower_layers": 2,
  "s2_hard_neg_k": 128,
  "eval_ks": [
    5,
    10,
    20
  ],
  "num_workers": 2,
  "s3_encoder": "BAAI/bge-reranker-v2-m3",
  "s3_pretrained_reranker": true,
  "s3_epochs": 3,
  "s3_batch_size": 32,
  "s3_grad_accum": 2,
  "s3_lr": 2e-05,
  "s3_warmup

### Train Stage 1 (item tower)

In [ ]:
cfg["s1_epochs"] = 5
cfg["s2_epochs"] = 3

In [44]:
s1_model = ContrastiveEncoder(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)

s1_model = train_stage1(
    s1_model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

torch.save(s1_model.tower.state_dict(), output_dir / 'stage1_encoder.pt')
print('Stage 1 complete')

02:09:24  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02:09:24  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/835193815a3936a24a0ee7dc9e3d48c1fbb19c55/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1224.50it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
02:09:24  INFO      ============================================================
02:09:24  INFO      Stage 1: Contrastive encoder fine-tuning
02:09:24  INFO      ============================================

[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled


KeyboardInterrupt: 

### Train Stage 2 (user tower)

In [ ]:
# Build CF mappings
item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
cf_dim = cfg.get('cf_dim', 0)
if cf_dim == 0:
    n_items_cf = 0
    n_users_cf = 0

model = TwoTowerModel(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    nhead=cfg['nhead'],
    temperature=cfg['temperature'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
    n_items=n_items_cf,
    n_users=n_users_cf,
    cf_dim=cf_dim,
    user_tower_layers=cfg.get('user_tower_layers', 1),
)

state = torch.load(output_dir / 'stage1_encoder.pt', map_location='cpu')
model.item_tower.load_state_dict(state, strict=False)
print('Stage 1 weights loaded into item tower')

model, best_val = train_stage2(
    model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

01:28:46  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
01:28:46  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/835193815a3936a24a0ee7dc9e3d48c1fbb19c55/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1052.30it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnin

[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled


01:28:47  INFO      ============================================================
01:28:47  INFO      Stage 2: Two-tower training
01:28:47  INFO      ============================================================


Stage 1 weights loaded into item tower


01:29:20  INFO      ItemTower + item CF frozen for Stage 2 — training UserTower + user CF only
01:29:20  INFO        Building item embedding cache (one-time, ItemTower frozen)...
01:29:36  INFO      S2 E1/3  step 100/867  loss=5.3999  lr=9.75e-06
01:29:38  INFO      S2 E1/3  step 200/867  loss=5.3635  lr=1.65e-05
01:29:40  INFO      S2 E1/3  step 300/867  loss=5.3334  lr=2.33e-05
/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:232: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
01:29:42  INFO      S2 E1/3  step 400/867  loss=5.3307  lr=3.00e-05


In [70]:
print('VAL METRICS')
print(format_metrics(best_val))

VAL METRICS
HR@10: 0.0302  HR@20: 0.0425  HR@5: 0.0201  MRR: 0.0182  NDCG@10: 0.0183  NDCG@20: 0.0214  NDCG@5: 0.0152


### Save model

In [ ]:
final_dir = pathlib.Path('checkpoints/new_new_model_cf_v1')
final_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), final_dir / 'model.pt')
tokenizer.save_pretrained(str(final_dir))
with open(final_dir / 'config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Model saved to {final_dir}')

Model saved to checkpoints/new_new_model_cf_v1


## Evaluate model

In [71]:
model.eval()

train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_epoch(
    model, tokenizer, test_df, anime_df,
    cfg=cfg, device=device, ks=[10, 20, 50],
    train_df=train_and_val,
)

print('TEST METRICS')
print(format_metrics(test_metrics))

02:55:12  INFO      Evaluation: 884 / 895 users have context embeddings


TEST METRICS
HR@10: 0.0291  HR@20: 0.0469  HR@50: 0.0726  MRR: 0.0165  NDCG@10: 0.0163  NDCG@20: 0.0208  NDCG@50: 0.0259


## Reranker model

In [ ]:
use_reranker_hpo = False

### Load checkpoint if it exists

In [ ]:
# cfg = {**DEFAULT_CFG}

# s3_tokenizer = AutoTokenizer.from_pretrained(cfg["s3_encoder"])

# reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
# reranker_model.load_state_dict(
#     torch.load('checkpoints/stage3_best.pt', map_location='cpu')
# )
# reranker_model.to(device).eval()
# print("Reranker loaded")

### Train reranker with HPO

In [ ]:
from model.train import run_hpo_reranker, DEFAULT_CFG

if use_reranker_hpo:
    best_s3_params = run_hpo_reranker(
        two_tower_model=model,
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=cfg,
        output_dir=output_dir,
        device=device,
        n_trials=20,
    )
else:
    best_s3_params = {}

In [56]:
DEFAULT_CFG["s3_epochs"] = 1

In [59]:
cfg_s3 = {**DEFAULT_CFG, **best_s3_params}

s3_tokenizer = AutoTokenizer.from_pretrained(cfg_s3["s3_encoder"])
reranker_model = train_stage3(
    CrossEncoderReranker(encoder_name=cfg['s3_encoder']),
    s3_tokenizer, train_df, val_df, anime_df,
    cfg=cfg_s3, output_dir=output_dir, device=device,
)

02:20:50  INFO      HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02:20:50  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/config.json "HTTP/1.1 200 OK"
02:20:51  INFO      HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
02:20:51  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/tokenizer_config.json "HTTP/1.1 200 OK"
02:20:51  INFO      HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
02:20:51  INFO      HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main?recursive=true&expand=false "HTTP/

### Save reranker model

In [60]:
torch.save(reranker_model.state_dict(), final_dir / 'reranker.pt')
s3_tokenizer.save_pretrained(str(final_dir / 'reranker_tokenizer'))

print(f'Reranker saved to {final_dir}')

Reranker saved to checkpoints/new_new_model_cf_v1


In [61]:
recommender = TwoTowerWithReranker(
    two_tower=model,
    reranker=reranker_model,
    tokenizer=tokenizer,
    reranker_tokenizer=s3_tokenizer,
    anime_df=anime_df,
    device=device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
    user_id_to_cf_idx=user_id_to_cf_idx if model.has_cf else None,
)

02:43:05  INFO      Building item embedding catalogue...
02:43:12  INFO      Catalogue ready: 9950 items × 256 dims


### Evaludate reranker

In [ ]:
val_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=val_df,
    train_df=train_df,
    ks=[10, 20, 5],
    retrieval_k=100,
)
print("RERANKER VAL METRICS")
print(format_metrics(val_metrics))

02:43:25  INFO      Evaluating reranker on 895 users (retrieval_k=100)
02:43:26  INFO        32 / 895 users evaluated...
02:43:34  INFO        352 / 895 users evaluated...
02:43:41  INFO        672 / 895 users evaluated...
02:43:46  INFO      Retrieval recall: 72 / 895 users had target in candidates (8.04%)


RERANKER VAL METRICS
HR@10: 0.0335  HR@20: 0.0425  HR@50: 0.0615  MRR: 0.2159  NDCG@10: 0.0200  NDCG@20: 0.0223  NDCG@50: 0.0260  retrieval_recall: 0.0804


In [65]:
# Test set (context = train + val)
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=test_df,
    train_df=train_and_val,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER TEST METRICS")
print(format_metrics(test_metrics))

02:48:28  INFO      Evaluating reranker on 895 users (retrieval_k=100)
02:48:29  INFO        32 / 895 users evaluated...
02:48:37  INFO        352 / 895 users evaluated...
02:48:45  INFO        672 / 895 users evaluated...
02:48:51  INFO      Retrieval recall: 86 / 895 users had target in candidates (9.61%)


RERANKER TEST METRICS
HR@10: 0.0402  HR@20: 0.0503  HR@50: 0.0782  MRR: 0.2103  NDCG@10: 0.0236  NDCG@20: 0.0261  NDCG@50: 0.0316  retrieval_recall: 0.0961


In [64]:
# Test set (context = train + val)
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=test_df,
    train_df=train_and_val,
    ks=[10, 20, 50],
    retrieval_k=500,
)
print("RERANKER TEST METRICS")
print(format_metrics(test_metrics))

02:45:00  INFO      Evaluating reranker on 895 users (retrieval_k=500)


02:45:10  INFO        32 / 895 users evaluated...
02:46:25  INFO        352 / 895 users evaluated...
02:47:36  INFO        672 / 895 users evaluated...
02:48:27  INFO      Retrieval recall: 252 / 895 users had target in candidates (28.16%)


RERANKER TEST METRICS
HR@10: 0.0704  HR@20: 0.0961  HR@50: 0.1341  MRR: 0.1119  NDCG@10: 0.0374  NDCG@20: 0.0438  NDCG@50: 0.0514  retrieval_recall: 0.2816


# Load fully trained model

In [ ]:
# final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# with open(final_dir / 'config.json') as f:
#     cfg = json.load(f)

# tokenizer = AutoTokenizer.from_pretrained(str(final_dir))

# # Rebuild CF mappings for the loaded model
# anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
# train_df = pd.read_parquet('processed_data/train_interactions.parquet')
# item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
# cf_dim = cfg.get('cf_dim', 0)
# if cf_dim == 0:
#     n_items_cf = 0
#     n_users_cf = 0

# model = TwoTowerModel(
#     encoder_name='intfloat/multilingual-e5-base',
#     proj_dim=cfg['proj_dim'], nhead=cfg['nhead'],
#     temperature=cfg['temperature'], dropout=cfg['dropout'],
#     freeze_mode=cfg['freeze_mode'], lora_rank=cfg['lora_rank'],
#     lora_alpha=cfg.get('lora_alpha', 16.0),
#     n_items=n_items_cf,
#     n_users=n_users_cf,
#     cf_dim=cf_dim,
#     user_tower_layers=cfg.get('user_tower_layers', 1),
# )
# model.load_state_dict(torch.load(final_dir / 'model.pt', map_location='cpu'), strict=False)
# model.to(device).eval()

# s3_tokenizer = AutoTokenizer.from_pretrained(str(final_dir / 'reranker_tokenizer'))
# reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
# reranker_model.load_state_dict(torch.load(final_dir / 'reranker.pt', map_location='cpu'))
# reranker_model.to(device).eval()

# print('Loaded')

# Inference 

In [ ]:
# sample user

user_history = [
    (5114, 10),   # FMA Brotherhood
    (9253, 9),   # Steins;Gate
    (11061, 9),   # HxH 2011
    (37510, 8),   # Mob Psycho 100 II
    (31240, 3),   # Re:Zero
]
top_k = 20

## Test model without reranker

In [67]:
model.eval()

item_matrix, id_to_idx = build_item_embedding_cache(
    model, tokenizer, anime_df, cfg, device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
)
idx_to_id = {v: k for k, v in id_to_idx.items()}

context_ids = [aid  for aid, _ in user_history if aid in id_to_idx]
context_scores = [s/10 for aid, s  in user_history if aid in id_to_idx]
watched_ids = set(context_ids)

ctx_idxs = torch.tensor([id_to_idx[a] for a in context_ids], dtype=torch.long)
ctx_embs = item_matrix[ctx_idxs].unsqueeze(0).to(device)
ctx_scores= torch.tensor([context_scores], dtype=torch.float).to(device)
ctx_mask  = torch.ones(1, len(context_ids), dtype=torch.bool).to(device)

with torch.no_grad():
    # No user_idx for anonymous/cold-start user — falls back to content-only
    user_vec = model.encode_user(ctx_embs, ctx_scores, ctx_mask)
    scores = (user_vec @ item_matrix.to(device).T).squeeze(0)

# Exclude watched items
watched_idxs = set(id_to_idx[a] for a in watched_ids if a in id_to_idx)
for idx in watched_idxs:
    scores[idx] = -1e9

top_idxs = scores.topk(top_k).indices.cpu().tolist()
top_ids = [idx_to_id[i] for i in top_idxs]

print(f'Top-{top_k} recommendations for this user:\n')
id_to_row = anime_df.set_index('id')
for rank, aid in enumerate(top_ids, 1):
    row = id_to_row.loc[aid]
    genres = ', '.join(row['genre_names'][:3])
    print(f'  {rank:2}. {row["name"]:<45} [{genres}]  score={row["score_global"]:.2f}')

Top-20 recommendations for this user:

   1. Shingeki no Kyojin Season 3                   [Action, Drama, Shounen]  score=0.85
   2. Shingeki no Kyojin                            [Action, Drama, Shounen]  score=0.84
   3. Shingeki no Kyojin: The Final Season Part 2   [Action, Drama, Shounen]  score=0.86
   4. Kimi no Na wa.                                [Drama, Award Winning]  score=0.87
   5. Ansatsu Kyoushitsu 2nd Season                 [Action, Comedy, School]  score=0.83
   6. Kizumonogatari II: Nekketsu-hen               [Action, Mystery, Vampire]  score=0.84
   7. Kono Subarashii Sekai ni Shukufuku wo! Movie: Kurenai Densetsu [Adventure, Comedy, Fantasy]  score=0.82
   8. Shingeki no Kyojin: Chronicle                 [Action, Drama, Shounen]  score=0.76
   9. Shingeki no Kyojin Season 2                   [Action, Drama, Shounen]  score=0.84
  10. Kono Subarashii Sekai ni Shukufuku wo! 2      [Adventure, Comedy, Fantasy]  score=0.80
  11. Dead Mount Death Play                   

## Test model with reranker

In [68]:
recs = recommender.recommend(user_history, top_k=top_k, retrieval_k=100)

print(f'Top- recommendations\n')
print(f'  {"#":<4} {"Title":<45} {"Reranker":>10} {"Retrieved":>10} {"Genres"}')
print('  ' + '-'*85)
for r in recs:
    genres = ', '.join(list(r['genres'])[:3]) if len(r['genres']) > 0 else '—'
    moved = r['two_tower_rank'] - r['rank']
    arrow = f'↑{moved}' if moved > 0 else (f'↓{-moved}' if moved < 0 else '=')
    print(f"  {r['rank']:<4} {r['name']:<45} {r['reranker_score']:>6.3f}  "
          f"#{r['two_tower_rank']:<3} {arrow:<5}  [{genres}]")

Top- recommendations

  #    Title                                           Reranker  Retrieved Genres
  -------------------------------------------------------------------------------------
  1    Death Note                                     0.802  #89  ↑88    [Shounen, Supernatural, Psychological]
  2    Grand Blue                                     0.786  #30  ↑28    [Comedy, Seinen, Adult Cast]
  3    Kimi no Na wa.                                 0.771  #8   ↑5     [Drama, Award Winning]
  4    Made in Abyss: Retsujitsu no Ougonkyou         0.765  #53  ↑49    [Adventure, Mystery, Drama]
  5    Shingeki no Kyojin: The Final Season           0.761  #42  ↑37    [Action, Drama, Shounen]
  6    Shingeki no Kyojin Season 3                    0.753  #15  ↑9     [Action, Drama, Shounen]
  7    Alice to Therese no Maboroshi Koujou           0.753  #27  ↑20    [Drama, Award Winning]
  8    Chainsaw Man                                   0.744  #37  ↑29    [Action, Fantasy, Shounen]
  9  

In [69]:
recs = recommender.recommend(user_history, top_k=top_k, retrieval_k=500)

print(f'Top- recommendations\n')
print(f'  {"#":<4} {"Title":<45} {"Reranker":>10} {"Retrieved":>10} {"Genres"}')
print('  ' + '-'*85)
for r in recs:
    genres = ', '.join(list(r['genres'])[:3]) if len(r['genres']) > 0 else '—'
    moved = r['two_tower_rank'] - r['rank']
    arrow = f'↑{moved}' if moved > 0 else (f'↓{-moved}' if moved < 0 else '=')
    print(f"  {r['rank']:<4} {r['name']:<45} {r['reranker_score']:>6.3f}  "
          f"#{r['two_tower_rank']:<3} {arrow:<5}  [{genres}]")

Top- recommendations

  #    Title                                           Reranker  Retrieved Genres
  -------------------------------------------------------------------------------------
  1    Hunter x Hunter                                0.890  #379 ↑378   [Action, Adventure, Fantasy]
  2    Hunter x Hunter: Greed Island                  0.842  #180 ↑178   [Action, Adventure, Fantasy]
  3    Death Note                                     0.802  #89  ↑86    [Shounen, Supernatural, Psychological]
  4    Dr. Stone: Stone Wars                          0.794  #143 ↑139   [Adventure, Comedy, Shounen]
  5    Sen to Chihiro no Kamikakushi                  0.790  #320 ↑315   [Adventure, Mythology, Fantasy]
  6    Shingeki no Kyojin                             0.788  #141 ↑135   [Action, Drama, Shounen]
  7    Grand Blue                                     0.786  #30  ↑23    [Comedy, Seinen, Adult Cast]
  8    Chainsaw Man Movie: Reze-hen                   0.782  #140 ↑132   [Action, Fan